In [15]:
from flask import Flask, request, jsonify
import csv
from datetime import datetime
import logging

app = Flask(__name__)

# File path for saving the data
csv_file = 'sensor_data.csv'

# Initialize logging
logging.basicConfig(level=logging.INFO)

# Initialize the CSV file with headers if it doesn't exist
def initialize_csv():
    try:
        with open(csv_file, mode='x', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(['AccX', 'AccY', 'AccZ', 'GyroX', 'GyroY', 'GyroZ', 'TimestampInMilliseconds'])
    except FileExistsError:
        pass

def convert_to_milliseconds(timestamp: str) -> int:
    """
    Convert timestamp in 'hh:mm:ss.mmm' format to milliseconds since midnight.
    """
    try:
        # Split the timestamp into hours, minutes, seconds, and milliseconds
        time_parts = timestamp.split(':')
        if len(time_parts) != 3:
            raise ValueError("Timestamp format should be 'hh:mm:ss.mmm'")

        hours = int(time_parts[0])
        minutes = int(time_parts[1])
        seconds, milliseconds = time_parts[2].split('.')
        seconds = int(seconds)
        milliseconds = int(milliseconds)

        # Convert to total milliseconds
        total_milliseconds = (hours * 3600 + minutes * 60 + seconds) * 1000 + milliseconds
        return total_milliseconds
    except Exception as e:
        raise ValueError(f"Error converting timestamp '{timestamp}': {e}")

@app.route('/upload-sensor-data', methods=['POST'])
def upload_sensor_data():
    try:
        data = request.get_json()
        gyro_data = data.get('gyro', [])
        acc_data = data.get('acc', [])

        # Validate data length
        if not gyro_data or not acc_data or len(gyro_data) != len(acc_data):
            return jsonify({"error": "Invalid or mismatched data"}), 400

        # Write data to CSV
        with open(csv_file, mode='a', newline='') as file:
            writer = csv.writer(file)
            for gyro, acc in zip(gyro_data, acc_data):
                # Extract timestamp from the data (assuming all points have the same timestamp)
                timestamp = gyro.get('timestamp') or acc.get('timestamp')
                if not timestamp:
                    return jsonify({"error": "Missing timestamp in data"}), 400

                # Convert timestamp to total milliseconds since midnight
                total_milliseconds = convert_to_milliseconds(timestamp)

                # Write to CSV with the timestamp in milliseconds
                writer.writerow([
                    acc['x'], acc['y'], acc['z'],  # Accelerometer data
                    gyro['x'], gyro['y'], gyro['z'],  # Gyroscope data
                    total_milliseconds  # Store the timestamp in milliseconds
                ])

        logging.info("Data received and stored successfully")
        return jsonify({"message": "Data received and stored successfully"}), 200
    except Exception as e:
        logging.error(f"Error processing data: {e}")
        return jsonify({"error": str(e)}), 500

if __name__ == '__main__':
    initialize_csv()
    app.run(host='0.0.0.0', port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.18.217:5000
INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.18.217:5000
Press CTRL+C to quit
INFO:werkzeug:Press CTRL+C to quit
192.168.18.104 - - [11/Dec/2024 02:09:33] "POST /upload-sensor-data HTTP/1.1" 400 -
INFO:werkzeug:192.168.18.104 - - [11/Dec/2024 02:09:33] "POST /upload-sensor-data HTTP/1.1" 400 -
INFO:root:Data received and stored successfully
192.168.18.104 - - [11/Dec/2024 02:09:35] "POST /upload-sensor-data HTTP/1.1" 200 -
INFO:werkzeug:192.168.18.104 - - [11/Dec/2024 02:09:35] "POST /upload-sensor-data HTTP/1.1" 200 -
INFO:root:Data received and stored successfully
192.168.18.104 - - [11/Dec/2024 02:09:36] "POST /upload-sensor-data HTTP/1.1" 200 -
INFO:werkzeug:192.168.18.104 - - [11/